# Multi-Agent System for Root Cause Analysis with MLflow ChatAgent

This notebook demonstrates how to:
1. Build a multi-agent RCA system using LangGraph
2. Wrap it with MLflow `ChatAgent` for Databricks integration
3. Enable automatic authentication passthrough
4. Deploy to AI Playground and Model Serving

## Why MLflow ChatAgent?
- **Automatic Authentication**: No manual PAT management for Unity Catalog resources
- **AI Playground**: Instant testing UI and SME feedback
- **Standardized Format**: Compatible with Databricks Agent Framework
- **One-Line Deployment**: `agents.deploy()` just works
- **Native Streaming**: Real-time progress updates in Databricks UI

In [0]:
%pip install -r ../requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
import sys

from multiAgentSystem.deps import get_deps
from multiAgentSystem.config import LLM_ENDPOINT_NAME, MAX_OUTER_ITERATIONS, MAX_ANALYZE_PARSE_LOOPS

print(f"LLM Endpoint: {LLM_ENDPOINT_NAME}")
print(f"Max Iterations: {MAX_OUTER_ITERATIONS}")
print(f"Inner Interations: {MAX_ANALYZE_PARSE_LOOPS}")

In [0]:
"""MLflow ChatAgent wrapper for the RCA multi-agent system."""

import mlflow
from multiAgentSystem.chat_agent_wrapper import RCALangGraphChatAgent
from multiAgentSystem.graph import build_graph

# Build the LangGraph multi-agent system
multi_agent_graph = build_graph()

# Wrap with MLflow ChatAgent for Databricks compatibility
AGENT = RCALangGraphChatAgent(multi_agent_graph)

# Enable MLflow autologging and set as the deployment model
mlflow.langchain.autolog()
mlflow.models.set_model(AGENT)

print("✅ RCA ChatAgent initialized")
print("✅ MLflow autologging enabled")
print("✅ Agent ready for deployment")

In [0]:
# Test the ChatAgent with example messages
from mlflow.types.agent import ChatAgentMessage

# Create messages in ChatAgent format
messages = [
    ChatAgentMessage(
        role="user",
        content="""I'm having issues with Query ID 01f0a416-cb80-1228-9eda-e3118e89fd48.
        
        Task: Figure out what went wrong and why it went wrong, see if there is any underlying reason for it."""
    )
]

# Custom inputs (logs path)
custom_inputs = {
    "logs_path": "/Volumes/amruthcatalogtest/default/testsparklogs/00761119/Longer-Bad-00761119_spark/"
}

# Test non-streaming prediction
result = AGENT.predict(messages=messages, custom_inputs=custom_inputs)
print(result)

In [0]:
# Test streaming prediction
print("=== STREAMING OUTPUT ===\n")

for chunk in AGENT.predict_stream(messages=messages, custom_inputs=custom_inputs):
    print(chunk.delta.content)
    print("-" * 80)

## Log and Deploy the Agent

Now that we've tested the ChatAgent, we can log it to MLflow and deploy it to Databricks.

In [ ]:
%%writefile agent.py
"""
Deployable MLflow ChatAgent for Spark RCA Multi-Agent System.

This file is used for logging and deployment to Databricks Model Serving.
"""

import mlflow
from multiAgentSystem.chat_agent_wrapper import RCALangGraphChatAgent
from multiAgentSystem.graph import build_graph
from multiAgentSystem.config import LLM_ENDPOINT_NAME

# Build the multi-agent graph
multi_agent_graph = build_graph()

# Wrap with ChatAgent
AGENT = RCALangGraphChatAgent(multi_agent_graph)

# Enable autologging and set as deployment model
mlflow.langchain.autolog()
mlflow.models.set_model(AGENT)

In [ ]:
# Specify Databricks resources for automatic authentication passthrough
from mlflow.models.resources import (
    DatabricksServingEndpoint,
    DatabricksVolume,
)
from multiAgentSystem.config import LLM_ENDPOINT_NAME

# Define resources the agent needs access to
resources = [
    # LLM endpoint for reasoning, analysis, etc.
    DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME),
    
    # Unity Catalog Volumes where Spark logs are stored
    # TODO: Add specific volumes your agent needs access to
    # Example: DatabricksVolume(volume="catalog.schema.volume_name"),
]

print(f"✅ Configured resources for automatic authentication:")
for resource in resources:
    print(f"  - {resource}")

In [ ]:
# Create input example for model signature
input_example = {
    "messages": [
        {
            "role": "user",
            "content": "Analyze Spark job failure with executor losses and OOM errors."
        }
    ],
    "custom_inputs": {
        "logs_path": "/Volumes/catalog/schema/spark_logs/job_123/"
    }
}

print("✅ Input example created for model signature")

In [ ]:
# Log the agent as an MLflow model
import mlflow
from pkg_resources import get_distribution

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        artifact_path="agent",
        python_model="agent.py",  # The file we wrote above
        input_example=input_example,
        resources=resources,  # Enables automatic authentication passthrough
        extra_pip_requirements=[
            f"databricks-connect=={get_distribution('databricks-connect').version}"
        ],
    )

print(f"✅ Agent logged to MLflow")
print(f"   Run ID: {logged_agent_info.run_id}")
print(f"   Model URI: {logged_agent_info.model_uri}")

## Pre-Deployment Validation

Test the logged model before registering to Unity Catalog.

In [ ]:
# Validate the logged model works correctly
validation_result = mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/agent",
    input_data=input_example,
    env_manager="uv",  # Use uv for faster environment resolution
)

print("✅ Pre-deployment validation successful")
print(validation_result)

## Register to Unity Catalog

Register the validated model to Unity Catalog for deployment.

In [ ]:
# Set Unity Catalog as the model registry
mlflow.set_registry_uri("databricks-uc")

# TODO: Update these values for your Unity Catalog location
catalog = "your_catalog"
schema = "your_schema"
model_name = "spark_rca_agent"

UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# Register the model
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri,
    name=UC_MODEL_NAME
)

print(f"✅ Model registered to Unity Catalog")
print(f"   Model: {UC_MODEL_NAME}")
print(f"   Version: {uc_registered_model_info.version}")

## Deploy to Model Serving

Deploy the agent to Databricks Model Serving with automatic authentication.

In [ ]:
# Deploy the agent to Model Serving
from databricks import agents

deployment_info = agents.deploy(
    UC_MODEL_NAME,
    uc_registered_model_info.version,
    tags={"project": "spark-rca", "framework": "langgraph"},
    # No need for environment_vars - automatic authentication handles credentials!
)

print(f"✅ Agent deployed to Model Serving")
print(f"   Endpoint: {deployment_info.endpoint_name}")
print(f"   Status: {deployment_info.state}")
print(f"\n🎯 Next steps:")
print(f"   1. Test in AI Playground")
print(f"   2. Share with SMEs for feedback")
print(f"   3. Integrate into production applications")